# Exercises XP: Text Preprocessing, NER, POS, and Word2Vec

Ce notebook couvre le prétraitement de texte, la reconnaissance d'entités nommées (NER),
l'étiquetage grammatical (POS), et les embeddings de mots avec Word2Vec.

## Ce que vous allez apprendre
- Nettoyer et normaliser des avis bruts avec la tokenisation, la suppression des mots vides et la lemmatisation.
- Extraire des caractéristiques linguistiques avec la NER et le POS tagging.
- Entraîner un modèle Word2Vec simple et interpréter ses dimensions vectorielles.
- Visualiser les embeddings de mots pour analyser les voisinages sémantiques.

## Setup · Installation des bibliothèques
Exécutez cette cellule une seule fois pour installer spaCy, nltk, gensim, et les outils de visualisation.

In [ ]:
# Installation silencieuse des bibliothèques nécessaires
%pip install --quiet spacy nltk gensim matplotlib seaborn --upgrade

In [ ]:
# ─────────────────────────────────────────────
# TÉLÉCHARGEMENT DES RESSOURCES NLTK ET SPACY
# ─────────────────────────────────────────────
# NLTK a besoin de fichiers supplémentaires pour fonctionner :
# - punkt / punkt_tab : pour découper le texte en mots (tokenisation)
# - wordnet / omw-1.4 : base de données lexicale pour la lemmatisation
# - stopwords : liste de mots vides en anglais (the, is, and...)
# - averaged_perceptron_tagger* : modèle pour l'étiquetage POS
# - tagsets : définitions des étiquettes POS (NN = nom commun, etc.)

import nltk
from spacy.cli import download as spacy_download
import spacy

# Liste de toutes les ressources NLTK à télécharger
resources = [
    "punkt",                          # tokeniseur de base
    "punkt_tab",                       # version mise à jour du tokeniseur
    "wordnet",                         # pour la lemmatisation
    "omw-1.4",                         # complément multilingue de WordNet
    "stopwords",                       # mots vides (stop words)
    "averaged_perceptron_tagger",      # tagger POS (ancien)
    "averaged_perceptron_tagger_eng",  # tagger POS (anglais, version récente)
    "tagsets",                         # descriptions des étiquettes POS
]

# Téléchargement de chaque ressource (quiet=True = sans affichage verbeux)
for res in resources:
    nltk.download(res, quiet=True)

# Téléchargement du modèle spaCy pour l'anglais (petit modèle = en_core_web_sm)
# Ce modèle permet de reconnaître les entités nommées (NER)
spacy_download("en_core_web_sm")

# Chargement du pipeline spaCy en mémoire
nlp = spacy.load("en_core_web_sm")

# Affichage des composants actifs dans le pipeline spaCy
# (ex: tokenizer, tagger, parser, ner...)
print("Composants du pipeline spaCy :", nlp.pipe_names)

## Exercice 1 · Prétraitement de texte, NER et POS

In [ ]:
# ─────────────────────────────────────────────
# JEUX DE DONNÉES BRUTES (RAW DATA)
# ─────────────────────────────────────────────
# Ce dictionnaire contient 10 avis de restaurants en anglais.
# On va les nettoyer et les analyser dans les étapes suivantes.

data = {
    'Review': [
        "At McDonald's the food was ok and the service was bad.",
        "I would not recommend this Japanese restaurant to anyone.",
        "I loved this restaurant when I traveled to Thailand last summer.",
        "The menu of Loving has a wide variety of options.",
        "The staff was friendly and helpful at Google's employees restaurant.",
        "The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
        "I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.",
        "The sushi at Sushi Express is always fresh and flavorful.",
        "The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
        "The dessert selection at Sweet Treats is to die for!"
    ]
}

# On extrait simplement la liste des avis pour faciliter les manipulations
raw_reviews = data['Review']

# Affichage pour vérifier que les données sont bien chargées
print(f"Nombre d'avis : {len(raw_reviews)}")
print("Premier avis :", raw_reviews[0])

### 1.1 Construire `preprocess_text()`

Cette fonction va :
1. Mettre le texte en minuscules et le tokeniser (découper en mots)
2. Supprimer la ponctuation (virgules, points, etc.)
3. Supprimer les mots vides (stop words : the, is, and...)
4. Appliquer un lemmatiseur (ramener chaque mot à sa forme de base)
5. Retourner une chaîne de caractères nettoyée

In [ ]:
# ─────────────────────────────────────────────
# IMPORTS NÉCESSAIRES POUR LE PRÉTRAITEMENT
# ─────────────────────────────────────────────
import re          # expressions régulières (non utilisé ici mais utile en général)
import string      # contient string.punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
import nltk
from nltk.corpus import stopwords     # liste de mots vides
from nltk.stem import WordNetLemmatizer  # lemmatiseur

# ─────────────────────────────────────────────
# INITIALISATION DES OUTILS
# ─────────────────────────────────────────────

# Ensemble des mots vides en anglais (convertis en set pour une recherche rapide)
# Exemple : {'the', 'a', 'is', 'in', 'it', 'and', ...}
stop_words = set(stopwords.words("english"))

# Initialisation du lemmatiseur
# La lemmatisation ramène un mot à sa forme canonique
# Exemples : 'running' → 'run', 'dishes' → 'dish', 'was' → 'be'
lemmatizer = WordNetLemmatizer()


def preprocess_text(text: str) -> str:
    """
    Nettoie un texte brut en plusieurs étapes :
      1. Mise en minuscules + tokenisation
      2. Suppression de la ponctuation
      3. Suppression des mots vides (stop words)
      4. Lemmatisation
    
    Paramètre :
        text (str) : le texte brut à nettoyer
    
    Retourne :
        str : le texte nettoyé sous forme de chaîne de mots séparés par des espaces
    """

    # ÉTAPE 1 : Mise en minuscules et tokenisation
    # text.lower() → tout en minuscules (ex: 'McDonald' → 'mcdonald')
    # word_tokenize() → découpe le texte en liste de tokens (mots + ponctuation)
    # Ex: "Food was bad." → ['food', 'was', 'bad', '.']
    tokens = nltk.word_tokenize(text.lower())

    # ÉTAPE 2 : Suppression de la ponctuation
    # string.punctuation contient tous les caractères de ponctuation
    # On garde uniquement les tokens qui ne sont PAS de la ponctuation
    # Ex: ['food', 'was', 'bad', '.'] → ['food', 'was', 'bad']
    tokens = [token for token in tokens if token not in string.punctuation]

    # ÉTAPE 3 : Suppression des mots vides (stop words)
    # Les mots vides sont des mots très fréquents qui n'apportent pas de sens
    # Ex: ['food', 'was', 'bad'] → ['food', 'bad']  ('was' est un stop word)
    tokens = [token for token in tokens if token not in stop_words]

    # ÉTAPE 4 : Lemmatisation
    # On ramène chaque mot à sa forme de base (lemme)
    # Ex: 'dishes' → 'dish', 'restaurants' → 'restaurant'
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    # RETOUR : on rejoint les tokens avec des espaces pour reformer une phrase
    # Ex: ['food', 'bad'] → 'food bad'
    return " ".join(tokens)


# ─── TEST RAPIDE ─────────────────────────────
# Testons la fonction sur le premier avis
exemple = raw_reviews[0]
print("Texte brut   :", exemple)
print("Texte nettoyé:", preprocess_text(exemple))

### 1.2 Créer un jeu de données nettoyé
On applique `preprocess_text` à tous les avis et on conserve les deux versions côte à côte.

In [ ]:
# ─────────────────────────────────────────────
# APPLICATION DU PRÉTRAITEMENT SUR TOUS LES AVIS
# ─────────────────────────────────────────────

# On applique preprocess_text() à chaque avis de la liste raw_reviews
# Le résultat est une nouvelle liste : cleaned_reviews
# zip() nous permet d'itérer sur les deux listes en parallèle
cleaned_reviews = [preprocess_text(review) for review in raw_reviews]

# Affichage côte à côte : brut vs nettoyé
print("=" * 65)
print("COMPARAISON : Texte brut vs Texte nettoyé")
print("=" * 65)

for i, (raw, cleaned) in enumerate(zip(raw_reviews, cleaned_reviews)):
    print(f"\n[Avis {i+1}]")
    print(f"  BRUT    : {raw}")
    print(f"  NETTOYÉ : {cleaned}")

print("\n" + "=" * 65)
print(f"Total : {len(cleaned_reviews)} avis traités")

### 1.3 Reconnaissance d'Entités Nommées (NER)

La NER permet d'identifier automatiquement des entités comme :
- **ORG** : organisations (McDonald's, Google, Pizza Hut)
- **GPE** : lieux géopolitiques (Thailand, Japan)
- **DATE** : dates (last summer, Monday)
- **PERSON** : noms de personnes

On utilise le modèle spaCy `en_core_web_sm`.

In [ ]:
# ─────────────────────────────────────────────
# FONCTION DE RECONNAISSANCE D'ENTITÉS NOMMÉES
# ─────────────────────────────────────────────

def perform_ner(text: str):
    """
    Effectue la reconnaissance d'entités nommées (NER) sur un texte.
    Utilise le modèle spaCy 'en_core_web_sm'.

    Paramètre :
        text (str) : le texte à analyser

    Retourne :
        list of tuples : liste de (entité, étiquette)
        Exemple : [('McDonald\'s', 'ORG'), ('Thailand', 'GPE')]
    """

    # On passe le texte dans le pipeline spaCy
    # 'doc' est un objet riche qui contient tokens, entités, dépendances...
    doc = nlp(text)

    # doc.ents contient toutes les entités détectées dans le texte
    # Chaque entité a :
    #   - .text  : le texte brut de l'entité (ex: "McDonald's")
    #   - .label_: l'étiquette de l'entité (ex: "ORG")
    entities = [(ent.text, ent.label_) for ent in doc.ents]

    return entities


# ─── TEST ─────────────────────────────────────
# Testons sur le premier avis brut
print("Test NER sur :", raw_reviews[0])
print("Entités trouvées :", perform_ner(raw_reviews[0]))

### 1.4 Étiquetage Grammatical (POS Tagging)

Le POS (Part-Of-Speech) tagging assigne une catégorie grammaticale à chaque mot :
- **NN** : nom commun singulier (food, restaurant)
- **NNS** : nom commun pluriel (dishes, steaks)
- **VBD** : verbe au passé (was, loved)
- **JJ** : adjectif (fresh, terrible)
- **RB** : adverbe (always, not)

On utilise `nltk.pos_tag` avec la tokenisation NLTK.

In [ ]:
# ─────────────────────────────────────────────
# IMPORTS NÉCESSAIRES POUR LE POS TAGGING
# ─────────────────────────────────────────────
from nltk import pos_tag, word_tokenize


def perform_pos_tagging(text: str):
    """
    Effectue l'étiquetage grammatical (POS Tagging) sur un texte.
    Utilise le tokeniseur et le tagger de NLTK.

    Paramètre :
        text (str) : le texte à analyser

    Retourne :
        list of tuples : liste de (mot, étiquette_POS)
        Exemple : [('food', 'NN'), ('was', 'VBD'), ('bad', 'JJ')]
    """

    # ÉTAPE 1 : Tokenisation du texte
    # word_tokenize() découpe le texte en une liste de mots
    # Ex: "Food was bad." → ['Food', 'was', 'bad', '.']
    tokens = word_tokenize(text)

    # ÉTAPE 2 : Étiquetage POS
    # pos_tag() assigne une étiquette grammaticale à chaque token
    # Ex: ['Food', 'was', 'bad'] → [('Food', 'NN'), ('was', 'VBD'), ('bad', 'JJ')]
    pos_tags = pos_tag(tokens)

    return pos_tags


# ─── TEST ─────────────────────────────────────
# Testons sur le premier avis brut
print("Test POS sur :", raw_reviews[0])
print("Tags POS      :", perform_pos_tagging(raw_reviews[0]))

# Pour comprendre ce que signifie une étiquette :
# Exemple : qu'est-ce que 'NN' signifie ?
print("\nDéfinition de l'étiquette NN :")
nltk.help.upenn_tagset('NN')

### 1.5 Appliquer NER et POS sur texte brut vs texte nettoyé

Comparons les sorties pour comprendre comment le prétraitement impacte les analyses linguistiques.

In [ ]:
# ─────────────────────────────────────────────
# COMPARAISON NER : BRUT vs NETTOYÉ
# ─────────────────────────────────────────────

# On prend les 3 premiers avis pour l'analyse (pour ne pas surcharger l'affichage)
sample_raw = raw_reviews[:3]
sample_clean = cleaned_reviews[:3]

print("=" * 65)
print("NER SUR TEXTE BRUT")
print("=" * 65)
# On parcourt les avis bruts et on affiche les entités détectées
for i, text in enumerate(sample_raw):
    entities = perform_ner(text)
    print(f"\n[Avis {i+1}] {text}")
    if entities:
        # Affichage de chaque entité avec son étiquette
        for ent_text, ent_label in entities:
            print(f"  → '{ent_text}' [{ent_label}]")
    else:
        print("  → Aucune entité détectée")

print("\n" + "=" * 65)
print("NER SUR TEXTE NETTOYÉ")
print("=" * 65)
# On fait pareil sur les textes nettoyés
# Observation attendue : le prétraitement supprime les majuscules et la
# ponctuation, ce qui va DÉGRADER la NER (les entités sont moins bien détectées)
for i, text in enumerate(sample_clean):
    entities = perform_ner(text)
    print(f"\n[Avis {i+1}] {text}")
    if entities:
        for ent_text, ent_label in entities:
            print(f"  → '{ent_text}' [{ent_label}]")
    else:
        print("  → Aucune entité détectée")

print("\n⚠️  Conclusion NER : le texte nettoyé (minuscules, sans ponctuation)")
print("   donne de MOINS BONS résultats en NER car spaCy utilise la casse")
print("   pour identifier les noms propres. La NER fonctionne mieux sur texte brut !")

In [ ]:
# ─────────────────────────────────────────────
# COMPARAISON POS : BRUT vs NETTOYÉ
# ─────────────────────────────────────────────

print("=" * 65)
print("POS TAGS SUR TEXTE BRUT")
print("=" * 65)
# POS sur les avis bruts
for i, text in enumerate(sample_raw):
    tags = perform_pos_tagging(text)
    print(f"\n[Avis {i+1}] {text}")
    print(f"  Tags : {tags}")

print("\n" + "=" * 65)
print("POS TAGS SUR TEXTE NETTOYÉ")
print("=" * 65)
# POS sur les avis nettoyés
# Observation : les stop words et la ponctuation sont absents,
# donc les tags sont sur moins de mots, mais restent utiles pour l'analyse
for i, text in enumerate(sample_clean):
    tags = perform_pos_tagging(text)
    print(f"\n[Avis {i+1}] {text}")
    print(f"  Tags : {tags}")

# Pour rappel : quelques étiquettes POS fréquentes
print("\n" + "=" * 65)
print("Aide-mémoire des principales étiquettes POS :")
print("  NN  = Nom commun singulier  (food, restaurant)")
print("  NNS = Nom commun pluriel    (dishes, options)")
print("  NNP = Nom propre singulier  (McDonald, Google)")
print("  VBD = Verbe passé           (was, loved)")
print("  JJ  = Adjectif              (fresh, terrible)")
print("  RB  = Adverbe               (always, not)")
print("  DT  = Déterminant           (the, a)")
print("  IN  = Préposition           (at, in, to)")

## Exercice 2 · Visualisation des Word Embeddings

Les **word embeddings** sont des représentations numériques des mots.
Chaque mot est converti en un vecteur de nombres réels.
Les mots avec des sens proches devraient avoir des vecteurs proches dans l'espace.

### 2.1 Entraîner un modèle Word2Vec

Word2Vec apprend des vecteurs de mots en analysant leur contexte dans les phrases.
Il existe deux variantes :
- **CBOW** (sg=0) : prédit un mot à partir de son contexte
- **Skip-gram** (sg=1) : prédit le contexte à partir d'un mot

In [ ]:
# ─────────────────────────────────────────────
# IMPORT DU MODÈLE WORD2VEC
# ─────────────────────────────────────────────
from gensim.models import Word2Vec

# ─────────────────────────────────────────────
# ÉTAPE 1 : TOKENISATION DES AVIS NETTOYÉS
# ─────────────────────────────────────────────
# Word2Vec attend une liste de listes de mots (chaque phrase = liste de tokens)
# On utilise cleaned_reviews (déjà nettoyés) et on les découpe sur les espaces
# Exemple : "mcdonald food ok service bad" → ['mcdonald', 'food', 'ok', 'service', 'bad']

tokenized_reviews = [review.split() for review in cleaned_reviews]

# Affichage d'un exemple pour vérifier
print("Exemple de revue tokenisée :")
print(tokenized_reviews[0])
print(f"\nNombre de phrases : {len(tokenized_reviews)}")

# ─────────────────────────────────────────────
# ÉTAPE 2 : ENTRAÎNEMENT DU MODÈLE WORD2VEC
# ─────────────────────────────────────────────
w2v_model = Word2Vec(
    sentences=tokenized_reviews,  # nos phrases tokenisées
    vector_size=50,               # taille des vecteurs (50 dimensions par mot)
                                  # Plus grand = plus expressif mais besoin de plus de données
    window=3,                     # contexte : nombre de mots avant/après le mot cible
                                  # Ici, on regarde les 3 mots avant et les 3 après
    min_count=1,                  # fréquence minimale : on garde les mots vus au moins 1 fois
                                  # (utile ici car notre dataset est très petit)
    sg=0,                         # algorithme : 0 = CBOW, 1 = Skip-gram
    workers=1,                    # nombre de threads (1 = reproductible)
    seed=42                       # graine aléatoire pour la reproductibilité
)

print("\n✅ Modèle Word2Vec entraîné avec succès !")
print(w2v_model)

### 2.2 Inspecter les dimensions des embeddings

In [ ]:
# ─────────────────────────────────────────────
# INSPECTION DU MODÈLE WORD2VEC
# ─────────────────────────────────────────────

# Le vocabulaire du modèle = tous les mots uniques qu'il a vus
vocabulary = list(w2v_model.wv.index_to_key)

print("=" * 55)
print("INFORMATIONS SUR LE MODÈLE WORD2VEC")
print("=" * 55)

# Taille du vecteur : chaque mot est représenté par un vecteur de N nombres
print(f"\n• Dimension des vecteurs : {w2v_model.wv.vector_size}")
print("  → Chaque mot est représenté par 50 nombres réels.")
print("  → Ces 50 valeurs encodent le 'sens' du mot selon son contexte.")

# Taille du vocabulaire : nombre de mots uniques connus par le modèle
print(f"\n• Taille du vocabulaire : {len(vocabulary)} mots")
print(f"  → Mots connus : {vocabulary[:15]}...")

# Exemple : vecteur du mot 'restaurant'
if 'restaurant' in w2v_model.wv:
    vecteur = w2v_model.wv['restaurant']
    print(f"\n• Vecteur du mot 'restaurant' (50 valeurs) :")
    print(f"  {vecteur[:10]}...  (seulement les 10 premières valeurs)")

# Exemple : mots les plus similaires à 'restaurant'
if 'restaurant' in w2v_model.wv:
    try:
        similaires = w2v_model.wv.most_similar('restaurant', topn=3)
        print(f"\n• Mots les plus proches de 'restaurant' :")
        for mot, score in similaires:
            print(f"  → '{mot}' (similarité cosinus : {score:.4f})")
    except Exception:
        print("  (dataset trop petit pour calculer des similarités fiables)")

### 2.3 Visualiser les word embeddings

On ne peut pas visualiser 50 dimensions directement.
On utilise **PCA** (Analyse en Composantes Principales) pour réduire les vecteurs à 2 dimensions,
puis on les affiche sur un graphique scatter plot.

In [ ]:
# ─────────────────────────────────────────────
# IMPORTS POUR LA VISUALISATION
# ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA  # Pour réduire les dimensions


def plot_word_embeddings(model, words=None, title="Word Embeddings (PCA 2D)"):
    """
    Visualise les embeddings Word2Vec en 2D à l'aide de PCA.

    Le modèle a des vecteurs de 50 dimensions. On utilise PCA pour les
    projeter en 2 dimensions afin de les afficher sur un graphique.

    Paramètres :
        model  : le modèle Word2Vec entraîné
        words  : liste de mots à afficher (None = tous les mots du vocabulaire)
        title  : titre du graphique
    """

    # ─── ÉTAPE 1 : Sélection des mots à afficher ─────────────────
    # Si aucune liste de mots n'est fournie, on utilise tout le vocabulaire
    if words is None:
        words = list(model.wv.index_to_key)

    # On ne garde que les mots qui sont dans le vocabulaire du modèle
    words = [w for w in words if w in model.wv]

    if len(words) == 0:
        print("Aucun mot valide à afficher.")
        return

    # ─── ÉTAPE 2 : Récupération des vecteurs ─────────────────────
    # Pour chaque mot, on récupère son vecteur de 50 valeurs
    # On crée une matrice de forme (nb_mots × 50)
    vectors = np.array([model.wv[w] for w in words])

    # ─── ÉTAPE 3 : Réduction de dimension avec PCA ───────────────
    # PCA projette nos vecteurs 50D en vecteurs 2D
    # n_components=2 : on veut 2 dimensions (X et Y) pour le graphique
    pca = PCA(n_components=2, random_state=42)
    coords_2d = pca.fit_transform(vectors)

    # ─── ÉTAPE 4 : Création du graphique ─────────────────────────
    plt.figure(figsize=(14, 9))  # taille du graphique en pouces

    # Scatter plot : chaque point = un mot
    # coords_2d[:, 0] = coordonnées sur l'axe X (1ère composante principale)
    # coords_2d[:, 1] = coordonnées sur l'axe Y (2ème composante principale)
    plt.scatter(
        coords_2d[:, 0],  # axe X
        coords_2d[:, 1],  # axe Y
        c='steelblue',    # couleur des points
        alpha=0.7,        # transparence (0=invisible, 1=opaque)
        s=80,             # taille des points
        edgecolors='navy', linewidths=0.5  # contour des points
    )

    # ─── ÉTAPE 5 : Annotations ────────────────────────────────────
    # On ajoute le nom du mot à côté de chaque point
    for i, word in enumerate(words):
        plt.annotate(
            word,                          # texte à afficher
            xy=(coords_2d[i, 0], coords_2d[i, 1]),  # position du point
            xytext=(5, 5),                 # décalage du texte par rapport au point
            textcoords='offset points',    # unité du décalage
            fontsize=8,                    # taille de la police
            color='darkred'                # couleur du texte
        )

    # ─── ÉTAPE 6 : Finalisation du graphique ─────────────────────
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel(
        f"Composante principale 1\n"
        f"(explique {pca.explained_variance_ratio_[0]*100:.1f}% de la variance)",
        fontsize=10
    )
    plt.ylabel(
        f"Composante principale 2\n"
        f"(explique {pca.explained_variance_ratio_[1]*100:.1f}% de la variance)",
        fontsize=10
    )
    plt.grid(True, linestyle='--', alpha=0.4)  # grille légère
    plt.tight_layout()  # ajustement automatique des marges
    plt.show()

    # ─── ANALYSE ─────────────────────────────────────────────────
    print(f"\n📊 Variance expliquée par la PCA :")
    print(f"   Axe X (PC1) : {pca.explained_variance_ratio_[0]*100:.1f}%")
    print(f"   Axe Y (PC2) : {pca.explained_variance_ratio_[1]*100:.1f}%")
    print(f"   Total       : {sum(pca.explained_variance_ratio_)*100:.1f}% de l'information conservée")


# ─── APPEL DE LA FONCTION ─────────────────────────────────────────
# Visualisation de tous les mots du vocabulaire
print("Génération du graphique des word embeddings...")
plot_word_embeddings(w2v_model, title="Word Embeddings - Avis Restaurants (PCA 2D)")

In [ ]:
# ─────────────────────────────────────────────
# GRAPHIQUE 2 : ZOOM SUR DES MOTS SÉLECTIONNÉS
# ─────────────────────────────────────────────
# On sélectionne manuellement des mots thématiques pour mieux analyser
# si les mots sémantiquement proches sont effectivement proches dans l'espace

# Mots liés à la nourriture / restauration
food_words = ['food', 'pizza', 'pasta', 'sushi', 'steaks', 'dessert', 'dish']

# Mots liés aux sentiments / qualité
sentiment_words = ['terrible', 'amazing', 'delicious', 'fresh', 'excellent', 'bad', 'friendly']

# Lieux / entités
place_words = ['restaurant', 'mcdonald', 'google', 'thailand', 'japan']

# Combinaison de tous les mots d'intérêt
selected_words = food_words + sentiment_words + place_words

# On ne garde que les mots présents dans le vocabulaire du modèle
selected_words = [w for w in selected_words if w in w2v_model.wv]

print(f"Mots sélectionnés présents dans le vocabulaire : {selected_words}")
plot_word_embeddings(
    w2v_model,
    words=selected_words,
    title="Word Embeddings - Mots sélectionnés par thème (PCA 2D)"
)

# ─── ANALYSE ET INTERPRÉTATION ────────────────────────────────────
print("\n" + "=" * 65)
print("ANALYSE DES RÉSULTATS")
print("=" * 65)
print("""
❓ Les mots liés sont-ils proches les uns des autres ?

   → Avec seulement 10 avis, le modèle n'a pas assez de données
     pour apprendre des représentations sémantiques fiables.
     Les clusters ne seront donc PAS très clairs.

❓ Pourquoi les résultats peuvent sembler aléatoires ?

   1. Données insuffisantes : Word2Vec a besoin de millions de phrases
      pour apprendre des représentations de qualité.
   2. Vocabulaire limité : avec ~60 mots uniques, peu de contexte partagé.
   3. PCA en 2D : on perd beaucoup d'information (50D → 2D).

💡 Pour améliorer les résultats :
   - Utiliser un corpus beaucoup plus grand (ex: avis Yelp, millions d'entrées)
   - Utiliser un modèle pré-entraîné (ex: Google News Word2Vec, GloVe)
   - Essayer t-SNE au lieu de PCA pour de meilleures visualisations
""")

### 2.4 Aller plus loin

Quelques pistes d'amélioration à explorer :

In [ ]:
# ─────────────────────────────────────────────
# BONUS : VISUALISATION AVEC t-SNE
# ─────────────────────────────────────────────
# t-SNE (t-distributed Stochastic Neighbor Embedding) est souvent
# meilleure que PCA pour visualiser des embeddings de mots
# car elle préserve mieux les relations locales entre les points.

from sklearn.manifold import TSNE

# Récupération de tous les mots et leurs vecteurs
all_words = list(w2v_model.wv.index_to_key)
all_vectors = np.array([w2v_model.wv[w] for w in all_words])

# Note : t-SNE nécessite que n_components < n_samples
# Avec un petit dataset, on ajuste perplexity (doit être < nb_mots)
n_words = len(all_words)
perplexity = min(5, n_words - 1)  # perplexity doit être < nb_mots

print(f"Nombre de mots dans le vocabulaire : {n_words}")
print(f"Perplexity utilisée pour t-SNE : {perplexity}")

# Application de t-SNE (2 dimensions)
tsne = TSNE(
    n_components=2,   # on veut 2 dimensions pour le graphique
    random_state=42,  # reproductibilité
    perplexity=perplexity,
    n_iter=1000       # nombre d'itérations d'optimisation
)
coords_tsne = tsne.fit_transform(all_vectors)

# Tracé du graphique
plt.figure(figsize=(14, 9))
plt.scatter(
    coords_tsne[:, 0], coords_tsne[:, 1],
    c='tomato', alpha=0.7, s=80, edgecolors='darkred', linewidths=0.5
)

# Annotation de chaque point avec le mot correspondant
for i, word in enumerate(all_words):
    plt.annotate(
        word,
        xy=(coords_tsne[i, 0], coords_tsne[i, 1]),
        xytext=(5, 5), textcoords='offset points',
        fontsize=8, color='navy'
    )

plt.title("Word Embeddings - Visualisation t-SNE (meilleure pour les clusters)",
          fontsize=13, fontweight='bold')
plt.xlabel("Dimension t-SNE 1", fontsize=10)
plt.ylabel("Dimension t-SNE 2", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print("\n✅ Visualisation t-SNE terminée.")
print("   Sur un grand corpus, t-SNE révèle des clusters sémantiques très clairs.")

In [ ]:
# ─────────────────────────────────────────────
# BONUS : COMPARAISON CBOW vs SKIP-GRAM
# ─────────────────────────────────────────────
# On entraîne deux modèles avec des algorithmes différents
# et on compare les mots les plus similaires

# Modèle CBOW (sg=0) : Continuous Bag of Words
# Prédit le mot central à partir des mots du contexte
# → Rapide, bon pour les mots fréquents
model_cbow = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=50, window=3, min_count=1,
    sg=0,  # CBOW
    workers=1, seed=42
)

# Modèle Skip-gram (sg=1)
# Prédit les mots du contexte à partir du mot central
# → Plus lent, mais meilleur pour les mots rares
model_skipgram = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=50, window=3, min_count=1,
    sg=1,  # Skip-gram
    workers=1, seed=42
)

print("=" * 55)
print("COMPARAISON CBOW vs SKIP-GRAM")
print("=" * 55)

# Comparaison pour quelques mots clés
mots_test = ['restaurant', 'food', 'pizza']

for mot in mots_test:
    if mot in model_cbow.wv and mot in model_skipgram.wv:
        try:
            sim_cbow = model_cbow.wv.most_similar(mot, topn=2)
            sim_sg   = model_skipgram.wv.most_similar(mot, topn=2)

            print(f"\n'{mot}'")
            print(f"  CBOW      → {[m for m,_ in sim_cbow]}")
            print(f"  Skip-gram → {[m for m,_ in sim_sg]}")
        except Exception:
            print(f"  '{mot}' : pas assez de données pour calculer des similarités")

print("\n💡 Sur ce petit dataset, les différences sont minimes.")
print("   Sur un grand corpus, Skip-gram est souvent meilleur pour les mots rares.")